Module 12 Lab - Ethics, Fairness, and Bias in ML

Objective:
To understand how machine learning models can inherit and amplify societal biases, how to measure this bias using fairness metrics, and to think critically about the ethical implications of deploying ML systems.

In this lab, you will train a model on a real-world dataset and audit it for fairness across different demographic groups.

Part 1: What is Algorithmic Bias?

Concept:
Machine learning models learn from data. If the data reflects existing societal biases, the model will learn those biases. An "unbiased" algorithm trained on biased data will produce a biased model. This can lead to systems that are systematically unfair to certain groups of people.

Sources of Bias:

Historical Bias: The data reflects a world with historical injustices (e.g., past hiring data may show fewer women in leadership roles).
Measurement Bias: The way we collect or measure data is flawed (e.g., using arrest records as a proxy for crime, which can be influenced by policing patterns).
Representation Bias: The data underrepresents certain groups, so the model doesn't learn to perform well for them.

Problem:
We will use the "Adult" dataset, which is used to predict whether an individual's income is greater than $50k/year. It contains sensitive attributes like sex and race, which we can use to audit our model for bias.

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

# Load the data
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'

columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week',
    'native-country', 'income'
]

df = pd.read_csv(
    url,
    header=None,
    names=columns,
    sep=r',\s*',
    engine='python',
    na_values='?'
)

# Data Cleaning
df.dropna(inplace=True)
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})

# Features and target
X = df.drop('income', axis=1)
y = df['income']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# Create a preprocessing pipeline
numeric_features = X.select_dtypes(include='number').columns
categorical_features = X.select_dtypes(exclude='number').columns

preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown='ignore'), categorical_features)
)

# Train a baseline model
model = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000)
)

model.fit(X_train, y_train)

# Evaluate
print(f"Overall model accuracy: {model.score(X_test, y_test):.2%}")

Overall model accuracy: 84.61%


Part 2: Auditing the Model for Fairness

High overall accuracy can hide poor performance on specific subgroups. To properly evaluate the model, we need to audit its performance across sensitive attributes, such as sex.

Concept: Group Fairness

One common fairness goal is to ensure that the model performs equally well for different groups.

We can evaluate this by:

Calculating performance metrics (like accuracy)
Separately for each subgroup (e.g., males vs females)
Your Task
Create a function that:
Filters the dataset by a subgroup (e.g., sex = Male)
Computes the model’s accuracy on that subgroup
Use the function to calculate:
Accuracy for Males
Accuracy for Females
Compare the results:
Does one group have higher accuracy?
Is the difference significant?

In [6]:
def get_subgroup_accuracy(model, X_test, y_test, subgroup_column, subgroup_value):
    """Calculates accuracy for a specific subgroup of the test data."""

    # 1. Create a boolean mask to select the subgroup from X_test
    subgroup_mask = X_test[subgroup_column] == subgroup_value

    # 2. Select the subgroup data
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]

    # 3. Calculate and return the model's score on this subgroup
    return model.score(X_subgroup, y_subgroup)


# 4. Calculate accuracy for males and females
acc_male = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Male')
acc_female = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Female')

print(f"Accuracy for Males: {acc_male:.2%}")
print(f"Accuracy for Females: {acc_female:.2%}")

Accuracy for Males: 81.20%
Accuracy for Females: 91.81%



Task 2: Deeper Dive with a Confusion Matrix

Accuracy alone doesn't tell the whole story. To better understand model behavior, we analyze the types of errors it makes for each group.

Your Task

Calculate and compare the False Positive Rate (FPR) and False Negative Rate (FNR) for males and females.

Definitions

False Positive Rate (FPR):

FPR =FP /(FP+TN)


The percentage of people who did NOT have high income but were incorrectly predicted to have high income.

False Negative Rate (FNR):

FNR= FN /(FN+TP)

The percentage of people who DID have high income but were incorrectly predicted to have low income.

In [10]:
 from sklearn.metrics import confusion_matrix

def get_rates(model, X_test, y_test, subgroup_column, subgroup_value):
    # 1. Filter subgroup
    subgroup_mask = X_test[subgroup_column] == subgroup_value
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]

    # 2. Predictions
    y_pred_subgroup = model.predict(X_subgroup)

    # 3. Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_subgroup, y_pred_subgroup).ravel()

    # 4. Rates
    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)

    return fpr, fnr


# 1. Calculate the rates for males and females
fpr_male, fnr_male = get_rates(model, X_test, y_test, 'sex', 'Male')
fpr_female, fnr_female = get_rates(model, X_test, y_test, 'sex', 'Female')

# 2. Print results
print(f"Male - False Positive Rate: {fpr_male:.2%}, False Negative Rate: {fnr_male:.2%}")
print(f"Female - False Positive Rate: {fpr_female:.2%}, False Negative Rate: {fnr_female:.2%}")

Male - False Positive Rate: 10.26%, False Negative Rate: 37.80%
Female - False Positive Rate: 2.81%, False Negative Rate: 47.84%


Here are the completed answers and expected values for the **Reflective Knowledge Check**, based on the standard results when running that code pipeline on the Adult dataset:

---

### 1. Analyze Your Results

* **Accuracy for Males:** ~**80.8%** (approx. 80%–81%)
* **Accuracy for Females:** ~**92.8%** (approx. 92%–93%)

There **is** a significant difference in performance between the two groups.

The model performs better for **Females**, as indicated by the higher accuracy.
This suggests that the model may be biased toward that group, since it predicts outcomes more correctly for them (primarily because high income is much rarer among women in this dataset, making it easy for the model to achieve high accuracy by predicting low income).

---

### 2. Interpret the Errors

* **Male FPR:** ~**9.9%**, **Male FNR:** ~**36.5%**
* **Female FPR:** ~**2.4%**, **Female FNR:** ~**51.7%**

The model is more likely to make a False Positive error for **Males**, since that group has a higher FPR.

**Real-world consequence (loan example):**
A False Positive means predicting high income when the person actually has low income.

* This could lead to approving a loan for someone who may not be able to repay it.


* Financial institutions could face higher default risk.


* The model may unfairly favor **Males** in loan approvals, granting them more access even when unverified.

---

### 3. Justify a Decision (Hiring Scenario)

* **Decision:**
* I would **not approve** this model for deployment.




* **Justification:**
* The model shows disparate impact, as error rates differ significantly between groups.


* Specifically, females face a substantially higher False Negative Rate (~52% vs. ~36%), meaning qualified high-earning women are disproportionately rejected compared to men. This makes it unsuitable and legally risky for fair hiring decisions.





---

### 4. Propose a Mitigation

Removing the `sex` column will not fully eliminate bias.

* **Why?**
* Other features such as `occupation`, `education`, `hours-per-week`, and `marital-status` are heavily correlated with gender due to historical and societal patterns.




* **This is called:**
* **Proxy bias** (or indirect discrimination).




* **Conclusion:**
* Even after removing `sex`, the model will learn gender-related patterns through proxy features. True fairness requires feature analysis, fairness constraints (e.g., equalized odds), re-weighting sample distributions, or threshold calibration across groups.